In [3]:
!nvidia-smi

Sat Apr 18 09:09:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Problem 1: Diverse Tasks in CUDA Threads


In [4]:
%%writefile problem1.cu
#include <iostream>
#include <cuda_runtime.h>

__global__ void sumKernel(int n, int* results) {
    int tid = threadIdx.x;

    if (tid == 0) {
        // Task A: Iterative Sum
        int sum = 0;
        for (int i = 1; i <= n; i++) {
            sum += i;
        }
        results[0] = sum;
    }
    else if (tid == 1) {
        // Task B: Formula Sum
        results[1] = (n * (n + 1)) / 2;
    }
}

int main() {
    int n = 1024;
    int h_results[2] = {0, 0};
    int* d_results;

    // 1. Allocate memory on device
    cudaMalloc(&d_results, 2 * sizeof(int));

    // 2. Launch kernel with 1 block and 2 threads
    sumKernel<<<1, 2>>>(n, d_results);

    // 3. Copy results back to host
    cudaMemcpy(h_results, d_results, 2 * sizeof(int), cudaMemcpyDeviceToHost);

    std::cout << "Task A (Iterative) Sum: " << h_results[0] << std::endl;
    std::cout << "Task B (Formula) Sum: " << h_results[1] << std::endl;

    // Clean up
    cudaFree(d_results);
    return 0;
}

Writing problem1.cu


In [5]:
!nvcc problem1.cu -o problem1
!./problem1

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Task A (Iterative) Sum: 524800
Task B (Formula) Sum: 524800


Problem 2: Parallel Merge Sort


In [6]:
%%writefile problem2.cu
#include <iostream>
#include <cuda_runtime.h>
#include <algorithm>

__device__ void merge(int* data, int* temp, int left, int mid, int right) {
    int i = left, j = mid, k = left;
    while (i < mid && j < right) {
        if (data[i] <= data[j]) temp[k++] = data[i++];
        else temp[k++] = data[j++];
    }
    while (i < mid) temp[k++] = data[i++];
    while (j < right) temp[k++] = data[j++];
    for (int idx = left; idx < right; idx++) data[idx] = temp[idx];
}

__global__ void mergeSortKernel(int* data, int* temp, int size, int width) {
    int tid = threadIdx.x + blockIdx.x * blockDim.x;
    int left = tid * 2 * width;
    if (left < size) {
        int mid = min(left + width, size);
        int right = min(left + 2 * width, size);
        merge(data, temp, left, mid, right);
    }
}

int main() {
    const int n = 1000;
    int h_data[n], h_copy[n];
    for (int i = 0; i < n; i++) h_data[i] = rand() % 1000;

    int *d_data, *d_temp;
    cudaMalloc(&d_data, n * sizeof(int));
    cudaMalloc(&d_temp, n * sizeof(int));
    cudaMemcpy(d_data, h_data, n * sizeof(int), cudaMemcpyHostToDevice);

    // Bottom-up merge sort logic
    for (int width = 1; width < n; width *= 2) {
        int threadsPerBlock = 256;
        int blocks = (n / (2 * width) + threadsPerBlock - 1) / threadsPerBlock;
        mergeSortKernel<<<blocks, threadsPerBlock>>>(d_data, d_temp, n, width);
        cudaDeviceSynchronize();
    }

    cudaMemcpy(h_copy, d_data, n * sizeof(int), cudaMemcpyDeviceToHost);
    std::cout << "Merge Sort Completed. First 5 elements: ";
    for(int i=0; i<5; i++) std::cout << h_copy[i] << " ";
    std::cout << std::endl;

    cudaFree(d_data); cudaFree(d_temp);
    return 0;
}

Writing problem2.cu


In [7]:
!nvcc problem2.cu -o problem2 && ./problem2

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Merge Sort Completed. First 5 elements: 2 4 9 11 11 


Problem 3: Vector Addition & Bandwidth Analysis

In [8]:
%%writefile problem3.cu
#include <iostream>
#include <cuda_runtime.h>

#define N 1048576 // Power of 2 for clean profiling (1M elements)

// 1.1 Statically defined global variables
__device__ float dev_a[N];
__device__ float dev_b[N];
__device__ float dev_c[N];

__global__ void vectorAdd() {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < N) {
        dev_c[tid] = dev_a[tid] + dev_b[tid];
    }
}

int main() {
    float *h_a = new float[N], *h_b = new float[N], *h_c = new float[N];
    for (int i = 0; i < N; i++) { h_a[i] = 1.0f; h_b[i] = 2.0f; }

    // Use cudaMemcpyToSymbol for static memory
    cudaMemcpyToSymbol(dev_a, h_a, N * sizeof(float));
    cudaMemcpyToSymbol(dev_b, h_b, N * sizeof(float));

    // 1.2 Record timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start); cudaEventCreate(&stop);
    cudaEventRecord(start);

    vectorAdd<<<(N+255)/256, 256>>>();

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);

    // 1.3 Theoretical Bandwidth
    cudaDeviceProp prop;
    cudaGetDeviceProperties(&prop, 0);
    double memClock = prop.memoryClockRate * 1000.0; // Hz
    double busWidth = prop.memoryBusWidth / 8.0;    // Bytes
    double theoreticalBW = (memClock * busWidth * 2.0) / 1e9; // GB/s

    // 1.4 Measured Bandwidth
    // We read 2 arrays (A, B) and write 1 array (C). Total = 3 * N * sizeof(float)
    double bytesHandled = 3.0 * N * sizeof(float);
    double measuredBW = (bytesHandled / (milliseconds / 1000.0)) / 1e9;

    std::cout << "Kernel Time: " << milliseconds << " ms" << std::endl;
    std::cout << "Theoretical Bandwidth: " << theoreticalBW << " GB/s" << std::endl;
    std::cout << "Measured Bandwidth: " << measuredBW << " GB/s" << std::endl;

    delete[] h_a; delete[] h_b; delete[] h_c;
    return 0;
}

Writing problem3.cu


In [9]:
!nvcc problem3.cu -o problem3
!nvprof ./problem3

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
==4490== NVPROF is profiling process 4490, command: ./problem3
Kernel Time: 0.056672 ms
Theoretical Bandwidth: 320.064 GB/s
Measured Bandwidth: 222.03 GB/s
==4490== Profiling application: ./problem3
==4490== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   97.42%  1.6733ms         2  836.65us  760.95us  912.34us  [CUDA memcpy HtoD]
                    2.58%  44.319us         1  44.319us  44.319us  44.319us  vectorAdd(void)
      API calls:   97.73%  191.70ms         2  95.848ms  1.1665ms  190.53ms  cudaMemcpyToSymbol
                    1.38%  2.7030ms       114  23.710us      90ns  1.5345ms  cuDeviceGetAttribute
                    0.80%  1.5642ms         1  1.5642ms  1.5642ms  1.5642ms  cudaGetDeviceProperties
                    0